In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


In [2]:
df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (20000, 17)
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'subject_length', 'body_length']


In [3]:
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["ticket_text"] = (
    df["subject"] + " " + df["body"]
)

print(df["ticket_text"].head())

0    Unvorhergesehener Absturz der Datenanalyse-Pla...
1    Customer Support Inquiry Seeking information o...
2    Data Analytics for Investment I am contacting ...
3    Krankenhaus-Dienstleistung-Problem Ein Medien-...
4    Security Dear Customer Support, I am reaching ...
Name: ticket_text, dtype: str


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2164.41it/s]


Embedding model loaded!


In [5]:
embeddings = model.encode(
    df["ticket_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 625/625 [34:30<00:00,  3.31s/it]  

Embedding shape: (20000, 384)


In [6]:
import os

os.makedirs("../models/embeddings", exist_ok=True)

np.save(
    "../models/embeddings/ticket_embeddings.npy",
    embeddings
)

print("Embeddings saved!")

Embeddings saved!


In [1]:
import faiss
import numpy as np

print("FAISS version:", faiss.__version__)

FAISS version: 1.15.0


In [2]:
embeddings = np.load(
    "../models/embeddings/ticket_embeddings.npy"
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (20000, 384)


In [3]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

print("Embeddings normalized!")

Embeddings normalized!


In [4]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 20000


In [5]:
import os

os.makedirs("../models/embeddings", exist_ok=True)

faiss.write_index(
    index,
    "../models/embeddings/ticket_index.faiss"
)

print("FAISS index saved!")

FAISS index saved!


In [6]:
print("Embedding shape:", embeddings.shape)
print("Number of vectors:", index.ntotal)

Embedding shape: (20000, 384)
Number of vectors: 20000


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1872.94it/s]


Embedding model loaded!


In [10]:
# New ticket
new_subject = "VPN connection stopped working"

new_body = """
I am unable to connect to the company VPN after a recent
Windows update. I cannot access internal applications.
"""

new_ticket = new_subject + " " + new_body

# Convert new ticket into an embedding
new_embedding = embedding_model.encode(
    [new_ticket]
)

# Convert to float32
new_embedding = new_embedding.astype("float32")

# Normalize
faiss.normalize_L2(new_embedding)

print("New embedding shape:", new_embedding.shape)

New embedding shape: (1, 384)


In [11]:
k = 5

similarity_scores, indices = index.search(
    new_embedding,
    k
)

print("Indices:", indices)
print("Similarity scores:", similarity_scores)

Indices: [[12448 14824 10505  1395  8079]]
Similarity scores: [[0.65969497 0.6569427  0.62814033 0.626684   0.6120166 ]]


In [13]:
import pandas as pd

df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

print("Dataset loaded:", df.shape)

Dataset loaded: (20000, 17)


In [14]:
results = []

for rank, idx in enumerate(indices[0], start=1):
    results.append({
        "rank": rank,
        "similarity": float(similarity_scores[0][rank - 1]),
        "subject": df.iloc[idx]["subject"],
        "body": df.iloc[idx]["body"],
        "priority": df.iloc[idx]["priority"],
        "type": df.iloc[idx]["type"],
        "queue": df.iloc[idx]["queue"],
    })

results_df = pd.DataFrame(results)

results_df[[
    "rank",
    "similarity",
    "subject",
    "priority",
    "type",
    "queue"
]]

,rank,similarity,subject,priority,type,queue
0,1,0.659695,Problem with VPN,high,Problem,IT Support
1,2,0.656943,VPN Issue,high,Problem,IT Support
2,3,0.628140,Trouble with Connection via VPN Router,low,Problem,Customer Service
3,4,0.626684,Support for Recent Data Analysis Processes,high,Incident,Technical Support
4,5,0.612017,Required Assistance for Data Breach,low,Problem,Technical Support


In [15]:
for rank, idx in enumerate(indices[0], start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("Similarity:", round(float(similarity_scores[0][rank - 1]), 4))
    print("Subject:", df.iloc[idx]["subject"])
    print("Priority:", df.iloc[idx]["priority"])
    print("Type:", df.iloc[idx]["type"])
    print("Queue:", df.iloc[idx]["queue"])
    print("\nAnswer:")
    print(df.iloc[idx]["answer"])


RESULT 1
Similarity: 0.6597
Subject: Problem with VPN
Priority: high
Type: Problem
Queue: IT Support

Answer:
Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.

RESULT 2
Similarity: 0.6569
Subject: VPN Issue
Priority: high
Type: Problem
Queue: IT Support

Answer:
Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.

RESULT 3
Similarity: 0.6281
Subject: Trouble with Connection via VPN Router
Priority: low
Type: Problem
Queue: Customer Service

Answer:
Acknowledging the connectivity issues with the VPN router. The problem seems to be impacting our project management meeting sessions and we need it resolved promptly. To assist with troubleshooting, could you provide more details about the router configuration and any specific error messages you are encountering? Additionally, it would be beneficial to know the number of users connected to the router during these sessions. If needed

# RAG Evaluation and Filtering

In [20]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["ticket_text"] = df["subject"] + " " + df["body"]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index(
    "../models/embeddings/ticket_index.faiss"
)

print("Dataset:", df.shape)
print("FAISS vectors:", index.ntotal)

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2100.41it/s]


Dataset: (20000, 18)
FAISS vectors: 20000


In [21]:
def retrieve_similar_tickets(ticket_text, top_k=5):

    # Create embedding for new ticket
    query_embedding = embedding_model.encode(
        [ticket_text]
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search FAISS
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "similarity": float(scores[0][rank - 1]),
            "index": int(idx),
            "subject": df.iloc[idx]["subject"],
            "body": df.iloc[idx]["body"],
            "answer": df.iloc[idx]["answer"],
            "priority": df.iloc[idx]["priority"],
            "type": df.iloc[idx]["type"],
            "queue": df.iloc[idx]["queue"]
        })

    return pd.DataFrame(results)

In [22]:
new_ticket = """
VPN connection stopped working after a recent Windows update.
I cannot connect to the company VPN or access internal applications.
"""

results = retrieve_similar_tickets(
    new_ticket,
    top_k=5
)

results[[
    "rank",
    "similarity",
    "subject",
    "priority",
    "type",
    "queue"
]]

,rank,similarity,subject,priority,type,queue
0,1,0.639387,Problem with VPN,high,Problem,IT Support
1,2,0.634197,VPN Issue,high,Problem,IT Support
2,3,0.610613,Required Assistance for Data Breach,low,Problem,Technical Support
3,4,0.605242,Trouble with Connection via VPN Router,low,Problem,Customer Service
4,5,0.602512,Support for Recent Data Analysis Processes,high,Incident,Technical Support


In [23]:
faiss.IndexFlatIP()

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x0000020834C0D630> >

In [24]:
# Keep the top 2 most relevant historical tickets
relevant_results = results.head(2).copy()

print("Number of relevant results:", len(relevant_results))

print(
    relevant_results[
        ["rank", "similarity", "subject", "priority", "type", "queue"]
    ]
)

Number of relevant results: 2
   rank  similarity           subject priority     type       queue
0     1    0.639387  Problem with VPN     high  Problem  IT Support
1     2    0.634197         VPN Issue     high  Problem  IT Support


In [25]:
relevant_results = results.head(2).copy()

print("Number of relevant results:", len(relevant_results))

print(
    relevant_results[
        ["rank", "similarity", "subject", "priority", "type", "queue"]
    ]
)

Number of relevant results: 2
   rank  similarity           subject priority     type       queue
0     1    0.639387  Problem with VPN     high  Problem  IT Support
1     2    0.634197         VPN Issue     high  Problem  IT Support


In [26]:
def build_rag_context(results):

    context = ""

    for _, row in results.iterrows():

        context += f"""
Historical Incident:
Subject: {row['subject']}
Type: {row['type']}
Priority: {row['priority']}
Queue: {row['queue']}
Historical Resolution: {row['answer']}

"""

    return context

In [27]:
context = build_rag_context(relevant_results)

print(context)


Historical Incident:
Subject: Problem with VPN
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.


Historical Incident:
Subject: VPN Issue
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.




In [28]:
print("Number of relevant results:", len(relevant_results))
print(relevant_results[["rank", "similarity", "subject"]])

Number of relevant results: 2
   rank  similarity           subject
0     1    0.639387  Problem with VPN
1     2    0.634197         VPN Issue


In [29]:
print("Context length:", len(context))
print(repr(context))

Context length: 442
'\nHistorical Incident:\nSubject: Problem with VPN\nType: Problem\nPriority: high\nQueue: IT Support\nHistorical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.\n\n\nHistorical Incident:\nSubject: VPN Issue\nType: Problem\nPriority: high\nQueue: IT Support\nHistorical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.\n\n'


In [30]:
print(context)


Historical Incident:
Subject: Problem with VPN
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.


Historical Incident:
Subject: VPN Issue
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.




In [31]:
from IPython.display import display, Markdown

display(Markdown(context))


Historical Incident:
Subject: Problem with VPN
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.


Historical Incident:
Subject: VPN Issue
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.



In [32]:
print(results[[
    "rank",
    "similarity",
    "subject",
    "priority",
    "type",
    "queue"
]])

   rank  similarity                                     subject priority  \
0     1    0.639387                            Problem with VPN     high   
1     2    0.634197                                   VPN Issue     high   
2     3    0.610613         Required Assistance for Data Breach      low   
3     4    0.605242      Trouble with Connection via VPN Router      low   
4     5    0.602512  Support for Recent Data Analysis Processes     high   

       type              queue  
0   Problem         IT Support  
1   Problem         IT Support  
2   Problem  Technical Support  
3   Problem   Customer Service  
4  Incident  Technical Support  


In [33]:
relevant_results = results.copy()

print("Number of results:", len(relevant_results))
print(relevant_results[[
    "rank",
    "similarity",
    "subject",
    "priority",
    "type",
    "queue"
]])

Number of results: 5
   rank  similarity                                     subject priority  \
0     1    0.639387                            Problem with VPN     high   
1     2    0.634197                                   VPN Issue     high   
2     3    0.610613         Required Assistance for Data Breach      low   
3     4    0.605242      Trouble with Connection via VPN Router      low   
4     5    0.602512  Support for Recent Data Analysis Processes     high   

       type              queue  
0   Problem         IT Support  
1   Problem         IT Support  
2   Problem  Technical Support  
3   Problem   Customer Service  
4  Incident  Technical Support  


In [34]:
context = build_rag_context(relevant_results)

print(context)


Historical Incident:
Subject: Problem with VPN
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.


Historical Incident:
Subject: VPN Issue
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.


Historical Incident:
Subject: Required Assistance for Data Breach
Type: Problem
Priority: low
Queue: Technical Support
Historical Resolution: To <name>, I am here to help with your data breach issue. To better understand the situation, could you provide more information about the breach, including the date it happened and the affected systems? I would like to schedule a call, would you be available to speak at <tel_num> for a call at your convenience or do you prefer another time?


Historical Incident:
Subject: Trouble with Connection via VPN Router


In [36]:
print(results[[
    "rank",
    "similarity",
    "subject",
    "priority",
    "type",
    "queue"
]])

   rank  similarity                                     subject priority  \
0     1    0.639387                            Problem with VPN     high   
1     2    0.634197                                   VPN Issue     high   
2     3    0.610613         Required Assistance for Data Breach      low   
3     4    0.605242      Trouble with Connection via VPN Router      low   
4     5    0.602512  Support for Recent Data Analysis Processes     high   

       type              queue  
0   Problem         IT Support  
1   Problem         IT Support  
2   Problem  Technical Support  
3   Problem   Customer Service  
4  Incident  Technical Support  


# RAG + LLM Generation Stage

In [35]:
def create_rag_prompt(new_ticket, context):

    prompt = f"""
You are an AI enterprise IT support assistant.

Your task is to analyze the new support ticket and suggest a
practical resolution using the historical incidents provided below.

NEW TICKET:
{new_ticket}

HISTORICAL INCIDENTS:
{context}

INSTRUCTIONS:
1. Use the historical incidents as evidence.
2. Do not invent facts that are not supported by the evidence.
3. Give a clear and concise troubleshooting recommendation.
4. If the historical evidence is insufficient, say that additional
   investigation is required.
5. Do not copy personal information, phone numbers, names, or
   placeholders from historical answers.
6. Do not mention that you are an AI.

Provide the response in this format:

Issue:
<brief description>

Recommended Resolution:
<step-by-step resolution>

Evidence:
<briefly mention which historical incidents support the recommendation>
"""

    return prompt

In [37]:
new_ticket = """
VPN connection stopped working after a recent Windows update.
I cannot connect to the company VPN or access internal applications.
"""

In [38]:
prompt = create_rag_prompt(
    new_ticket,
    context
)

print(prompt)


You are an AI enterprise IT support assistant.

Your task is to analyze the new support ticket and suggest a
practical resolution using the historical incidents provided below.

NEW TICKET:

VPN connection stopped working after a recent Windows update.
I cannot connect to the company VPN or access internal applications.


HISTORICAL INCIDENTS:

Historical Incident:
Subject: Problem with VPN
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please contact us at <tel_num> to troubleshoot the VPN issue and resolve the connectivity problem.


Historical Incident:
Subject: VPN Issue
Type: Problem
Priority: high
Queue: IT Support
Historical Resolution: Please call us to investigate the VPN issue and resolve the connection problems. Our phone number is <tel_num>.


Historical Incident:
Subject: Required Assistance for Data Breach
Type: Problem
Priority: low
Queue: Technical Support
Historical Resolution: To <name>, I am here to help with your data breach issue. To better 

In [39]:
print("Prompt length:", len(prompt))

Prompt length: 3024


In [1]:
import ollama

In [40]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

Issue:
VPN connection stopped working after a recent Windows update, resulting in inability to connect to company VPN and access internal applications.

Recommended Resolution:
1. Please ensure that all Windows updates are completed.
2. Restart your VPN client application and then reconnect to the VPN server.
3. If restarting does not resolve the issue, try updating the VPN client application to the latest version.

Evidence:
Historical Incident: "Subject: Problem with VPN" and "Subject: Support for Recent Data Analysis Processes" suggest that a restart of the VPN connection may resolve connectivity issues.
